In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
df = pd.read_csv('/content/IMDB Dataset.csv')
df = df[:10000]
df.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [3]:
df.shape

(10000, 2)

In [4]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [5]:
df.isnull().sum()

,0
review,0
sentiment,0


# Text Preprocessing
Ideal Pipeline Order:
1. Remove HTML tags
2. Lowercasing
3. Remove punctuation
4. Remove numbers (optional but recommended)
5. Tokenization
6. Remove stopwords
7. Lemmatization (or Stemming)

In [6]:
from nltk.corpus import wordnet
lemmatizer = WordNetLemmatizer()
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stop_words.discard('not')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

# POS tag converter
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def Processing_Pipeline(text):
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 2. Lowercasing
    text = text.lower()
    # 3. Remove punctuation
    text = re.sub(r'[^​\w\s]', ' ', text)
    # 4. Tokenization
    tokens = word_tokenize(text)
    # 5. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # 6. POS tagging
    pos_tags = nltk.pos_tag(tokens)
    # 7. Lemmatization with POS
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    # 8. Join back
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
df['clean_review'] = df['review'].apply(Processing_Pipeline)

In [8]:
df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one reviewer mention watch 1 oz episode hook r...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,positive,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,negative,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter mattei love time money visually stunnin...


In [9]:
X = df['clean_review']
y = df['sentiment']

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
y

array([1, 1, 1, ..., 0, 0, 1])

# Text Representation

In [11]:
# BOW
cv = CountVectorizer(max_features=10000)
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

In [12]:
X_train_bow.shape

(8000, 10000)

In [13]:
y_train.shape

(8000,)

In [14]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train_bow, y_train)

LogisticRegression()

In [15]:
from sklearn.metrics import accuracy_score
y_pred = lr.predict(X_test_bow)
print(accuracy_score(y_test, y_pred))

0.85


Accuracy score using different models:
- GussianNB : 0.61
- LogisticRegression: 0.85
- RandomForestRegressor : 0.85

In [16]:
# Ngrams
cv = CountVectorizer(max_features=10000, ngram_range=(1,2))
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

lr = LogisticRegression()
lr.fit(X_train_bow, y_train)

y_pred = lr.predict(X_test_bow)
print(accuracy_score(y_test, y_pred))

0.8595


In [17]:
# Td-idf
from sklearn.feature_extraction.text import TfidfVectorizer
tdidf = TfidfVectorizer()

X_train_tdidf = tdidf.fit_transform(X_train).toarray()
X_test_tdidf = tdidf.transform(X_test).toarray()

lr = LogisticRegression()
lr.fit(X_train_tdidf, y_train)

y_pred = lr.predict(X_test_tdidf)
print(accuracy_score(y_test, y_pred))

0.867
